# Sugarcane explainability and bioethanol process screening
**Revision: 29 October 2026. Academic prototype; no industrial validation claimed.**

The original image-classification notebook is attributed below to its original authors. Retain that credit and check the source licence before redistribution. Your resume must describe your own implemented and understood contribution.

**Two independent parts:** Part A retains the leaf-classification study with evaluation fixes. Part B adds a chemical-engineering screening model for sugarcane-juice-derived fermentable sugar. This is a simplified **1G glucose-equivalent route**, not a 2G bagasse model. Disease predictions are NOT converted into sugar concentration, conversion, or ethanol yield: that relationship needs paired measurements.

**Run:** Part A needs the original image dataset, TensorFlow, scikit-learn, SHAP, Pillow and ImageNet weights. It has not been retrained here. All old outputs have been cleared because evaluation changed. For the immediately runnable process study, skip directly to Part B and run its cells in sequence; only Python, NumPy and pandas are needed.

**Changes to Part A:** independent stratified file splits, exact duplicate removal, correct MobileNetV2 scaling, enabled early stopping, and training-set SHAP backgrounds. SHAP visualisations still need verification with your installed versions. A random image split does not establish cross-farm generalisation.

This notebook was ceated by Nursultan Sheishekeyev, Mateusz Zbigniew Zalewski and Ferran Oliver Gerones for a special course at DTU.

# Sugarcane Leaf Disease data set 
**Dataset information** 

[Dataset](https://www.kaggle.com/datasets/nirmalsankalana/sugarcane-leaf-disease-dataset/data) from Kaggle. This is a dataset of manually collected images of sugarcane leaves. The images are classified according to five categories: Healthy, Mosaic, Redrot, Rust and Yellow disease. Therefore, the task is a multi-class classification problem. The data in a folder with 5 subfolders containgin images for each claass in each subfolder. 


Import libraries and packages

In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import re
import seaborn as sns
import tensorflow as tf
from glob import glob
from PIL import Image
from sklearn.metrics import classification_report
from tensorflow import keras

os.environ['KMP_DUPLICATE_LIB_OK']='True' #bug fixing

Data directory and class investigation

In [ ]:
data_dir="data/Sugarcane"
print(os.listdir(data_dir)) #print the different classes of sugarcane leafs in the folder

class_names=os.listdir(data_dir) #list of classes in the directory 
print(class_names) 

##show file names in of all 5 classes in the directory to see as a for loop. 

# for i in class_names: 
#     print(i)
#     print(os.listdir(data_dir+"/"+i))

In [ ]:
#Save directory for all images in a list
all_images=glob(data_dir+"/*/*.*")

#Scanning the list for image types by difining the "filetype" split by "." and taking the last element of the list
all_image_types=set([i.split(".")[-1] for i in all_images])
print(all_image_types)


#Scaning the data directories for classes for unique image name without numbering
keywords = ['healthy', 'mosaic', 'redrot', 'rust', 'yellow']

for i in class_names: 
    print(i)
    files = os.listdir(data_dir+"/"+i)
    for file in files:
        # Remove numbers from the file name
        file_without_numbers = re.sub(r'\d+', '', file)
        # Check if the file name without numbers is one of the keywords
        if file_without_numbers in keywords:
            print(file)


All images are .jpeg files and 5 classes ['Healthy', 'Mosaic', 'RedRot', 'Rust', 'Yellow'] of sugercane leafs stores in 5 subfolders

## Investigation of the images of Sugarcane leafs 
Based on the work from older notebooks made in this cource

Visualisation of images

In [ ]:
#number of desired images to show
num_images=3

#function to load random images from a class and add resolution information
def load_random_images(class_name, num_images):

    image_paths = glob(f"data/Sugarcane/{class_name}/*") #import images from directory 
    random_paths = random.sample(image_paths, min(num_images, len(image_paths))) # pick random pictures 
    images = [mpimg.imread(path) for path in random_paths] #generate image 
    resolutions = [Image.open(path).size for path in random_paths] #image resolution

    return images, class_name, resolutions


In [ ]:
# Plot three random images from each class 3x5
fig, axes = plt.subplots(len(class_names), num_images, figsize=(12, 18))

for row, class_name in zip(axes, class_names): #zip function to iterate over two lists at the same time
    imgs, label, resolutions = load_random_images(class_name, num_images) #function from cell above
    for ax, img, resolution in zip(row, imgs, resolutions): #zip function to iterate over three lists at the same time to extract image, label and resolution
        ax.imshow(img)
        ax.set_title(f"Class: {label}, Resolution: {resolution[0]} x {resolution[1]}")
        ax.axis('off')
        #for each ax in subplots plot, the image, label info and resolution info

plt.tight_layout() 
plt.show()


#5*3 subplot same procedure as above but working with columns and not rows per class. 
# fig, axes = plt.subplots(num_images, len(class_names), figsize=(12, 18))

# for col, class_name in zip(axes.T, class_names):
#     imgs, label, resolutions = load_random_images(class_name, num_images)
#     for ax, img, resolution in zip(col, imgs, resolutions):
#         ax.imshow(img)
#         ax.set_title(f"Class: {label}, Resolution: {resolution[0]} x {resolution[1]}")
#         ax.axis('off')

# plt.tight_layout()
# plt.show()

From the plot of images, it shows that the resolutions of images are different, for each random image for all classes. 
For image classification preprocessing may be needed due to the high difference in resolutions. 

**Investigation of the resolution of all images**

In [ ]:
# Lists to store resolutions for all images in each class
all_resolutions = []

#function to extract resolution data from images 
def load_all_images_resolution(images):
    resolutions = [] #empty list to store resolutions for each image

    for image_path in images:
        # Analyze image resolution
        resolution = Image.open(image_path).size #open image and extract resolution
        resolutions.append(resolution) #append resolution to list for each image
        all_resolutions.append(resolution) #append resolution to list for all images in all classes

    return resolutions




In [ ]:
# Load resolutions for all images

img_array = np.array(glob("data/Sugarcane/*/*")) #create np.array of all images in all classes
resolutions_per_class = {} #empty dictionary to store resolutions per class

for class_name in class_names: #iterate over all classes
    class_images = [image for image in img_array if class_name in image] #create list of images per class
    resolutions = load_all_images_resolution(class_images) #load resolution data for each image in each class
    resolutions_per_class[class_name] = resolutions #store resolution data per class in dictionary
    # Plot scatter plot for image resolutions within each class
    resolutions_np = np.array(resolutions)
    plt.scatter(resolutions_np[:, 0], resolutions_np[:, 1], label=f"Class: {class_name}")

# Plot scatter plot for image resolutions across all classes
all_resolutions_np = np.array(all_resolutions)
plt.xlabel("Width")
plt.ylabel("Height")
plt.title("Image Resolutions Across Classes")
plt.legend()
plt.show()

From the plot the resolutions of images have high differences in the format type. Important to standardise it for model classifier. 

**Investigation of the dimensions of the classes**

In [ ]:
#function to count images in classes
def count_images_in_class(class_name): #function to count images in each class
    images = np.array(glob(f"data/Sugarcane/{class_name}/*")) #create np.array of all images in all classes
    count = len(images) #count images in each class
    return count

In [ ]:
# Variables to store total count and count per class
total_count = 0 #initial count of images 
count_per_class = {} #empty dictionary to store count per class

# Count images for each class
for class_name in class_names: 
    count = count_images_in_class(class_name) #count images in each class
    count_per_class[class_name] = count #store count per class in dictionary
    total_count += count #add count per class to total count

# Print total count
print(f"Total number of images: {total_count}")

# Print count per class
print("Number of images per class:")
for class_name, count in count_per_class.items():
    print(f"{class_name}: {count}")

# Create a bar chart to visualize the count of different classes
fig, ax = plt.subplots()
ax.bar(count_per_class.keys(), count_per_class.values(), label='Per Class')
#ax.bar('Total', total_count, color='orange', label='Total')
ax.set_xlabel('Class')
ax.set_ylabel('Number of Images')
ax.set_title('Number of Images per Class')
ax.legend()
plt.show()

The results shows the classes are not balance. However since all classes are +-500, and the differences in class sizes quite small. We continue. 

## Model building 

https://www.geeksforgeeks.org/visualizing-representations-of-outputs-activations-of-each-cnn-layer/ 

https://www.kaggle.com/code/quadeer15sh/grad-cam-what-do-cnns-see 

https://keras.io/examples/vision/visualizing_what_convnets_learn/ 

https://keras.io/api/models/sequential/#sequential-class

https://www.geeksforgeeks.org/python-image-classification-using-keras/ 

https://www.tensorflow.org/tutorials/keras/overfit_and_underfit 

https://www.kaggle.com/code/uysimty/keras-cnn-dog-or-cat-classification 



In [ ]:
epochs = 50 #number of epochs for training - large epoch may leed to overfitting, but keep constant for all models
image_size = (128, 128)
input_shape = (128, 128, 3) #image size and color channels
dataset_dir = "data/Sugarcane"
seed = 123 #random seed for reproducibility
batch_size = 32 #batch size for training
buffer_size = 250 #buffer size for shuffling
fine_tune_point = 125 
learning_rate = 0.001 #learning rate for optimizer



In [ ]:
from sklearn.model_selection import train_test_split
from pathlib import Path
import hashlib
# Exact duplicates are excluded before splitting. Field/plant grouping still requires metadata.
paths = sorted(Path(dataset_dir).glob('*/*.jpeg'))
unique, seen = [], set()
for path in paths:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest not in seen:
        seen.add(digest); unique.append(path)
class_names = sorted({p.parent.name for p in unique})
labels = [class_names.index(p.parent.name) for p in unique]
tr, hold, yl_tr, yl_hold = train_test_split(
    [str(p) for p in unique], labels, test_size=0.30, stratify=labels, random_state=seed)
va, te, yl_va, yl_te = train_test_split(
    hold, yl_hold, test_size=0.50, stratify=yl_hold, random_state=seed)
assert not (set(tr) & set(va) or set(tr) & set(te) or set(va) & set(te))
def make_dataset(paths, labels):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    def decode(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(image, image_size), label
    return ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE).batch(batch_size)
train_ds = make_dataset(tr, yl_tr)
validation_ds = make_dataset(va, yl_va)
test_ds = make_dataset(te, yl_te)
print('Independent train/validation/test:', len(tr), len(va), len(te))

In [ ]:
#check the images shapes after preprocessing

class_names = sorted(class_names) #extract class names from training set
for example_image, example_label in train_ds.take(1): #take one image and label from training set
  break

print(f"Image Shape: {example_image.shape}")

Splits are now created at the file level before batching. The previous nested shard implementation did not establish an independent test set. Near-duplicates and images from the same plant still require group-aware splitting before any field-performance claim.

In [ ]:
train_ds = train_ds.cache().shuffle(buffer_size).prefetch(buffer_size=tf.data.AUTOTUNE) #cache and shuffle training set
validation_ds = validation_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE) #cache validation set
test_ds = test_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE) #cache test set

**Simple Model**

In [ ]:
#bulding a simple model with keras

simple_model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(128, 128, 3)), #flatten the input shape
    tf.keras.layers.Dense(8, activation='relu'), #dense layer with 8 neurons and relu activation function
    tf.keras.layers.Dense(len(class_names)) #dense layer with number of neurons equal 5 classes
])
#dummy input for model summary 
dummy_input = tf.random.normal(shape=(1, 128, 128, 3)) #create dummy input for model

simple_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
) #compile model with optimizer, loss function and metrics

# Call the model with a dummy input to trigger the build process
simple_model(dummy_input) 

# Now you can print the summary
simple_model.summary()



In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

#Define early stopping - not relevant when epochs are set to 50 for all models. But can be used in order to stop training when validation loss does not improve anymore. 
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=5,  # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value
)


simple_modelhistory = simple_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=epochs,
    callbacks=[early_stopping]
) #fit model to training data and validate on validation data

#save the model
simple_model.save('simple_model.h5')

In [ ]:
#accuary and loss plot
simple_model_metrics = simple_modelhistory.history #extract metrics from history
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.plot(simple_modelhistory.epoch, simple_model_metrics['loss'], simple_model_metrics['val_loss'])
plt.legend(['training', 'validation'])
plt.ylim([0, max(plt.ylim())])
plt.ylabel('Loss')
plt.xlabel('Epoch')

plt.subplot(1, 2, 2)
plt.plot(simple_modelhistory.epoch, simple_model_metrics['accuracy'], simple_model_metrics['val_accuracy'])
plt.legend(['training', 'validation'])
plt.ylim([0, 1])
plt.ylabel('Accuracy')
plt.xlabel('Epoch')

In [ ]:
simple_model.evaluate(test_ds, return_dict=True) #evaluate model on test data

y_pred_simple = simple_model.predict(test_ds) #predict labels for test data
y_pred_simple = np.argmax(y_pred_simple, axis=1) #extract predicted labels from test data
y_true_simple = np.concatenate([y for x, y in test_ds], axis=0) #extract true labels from test data

confusion_mtx = tf.math.confusion_matrix(y_true_simple, y_pred_simple) #create confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_mtx,
            xticklabels=class_names,
            yticklabels=class_names,
            annot=True, fmt='g')
plt.xlabel('Prediction')
plt.ylabel('Label')
plt.show()



print(classification_report(y_true_simple, y_pred_simple, target_names=class_names))

In [ ]:
#plot of 9 random images from test set with predicted and true label
plt.figure(figsize=(12, 9))
for images, labels in test_ds.take(1): #take one image and label from test set
    for i in range(9): #iterate over 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8")) #plot image
        color = 'green' if y_pred_simple[i] == labels[i] else 'red' #if predicted label is the same as true label, color is green, else red
        plt.title(f"Predicted: {class_names[y_pred_simple[i]]}, True: {class_names[labels[i]]}", color=color) #add predicted and true label to title
        plt.suptitle("Sugarcane Leafs (Healthy or Diseased)")
        plt.axis("off")

plt.show()

In [ ]:
import shap
import numpy as np

#Convert the TensorFlow dataset to numpy array
test_ds_numpy = np.concatenate([x for x, _ in test_ds], axis=0)
test_labels = np.concatenate([y for _, y in test_ds], axis=0)  # Get the true labels

#Select two images from the test set - just some random from the test set. 
selected_images = test_ds_numpy[8:10]
selected_labels = test_labels[8:10]  # Select the true labels for the selected images

#predictions for the selected images
predictions = simple_model.predict(selected_images)
predicted_classes = np.argmax(predictions, axis=1)  # Get the predicted classes

#shap explainer for the model predictions for selected_images
explainer = shap.GradientExplainer(simple_model, selected_images)

#shap values from explainer
shap_values, indexes = explainer.shap_values(selected_images, ranked_outputs=2)

#Plot shap values
shap.image_plot(shap_values, selected_images,show=False)


#Print the predicted and true classes
for i in range(len(selected_images)):
    print(f"Image {i+1}:")
    print(f"Predicted class: {predicted_classes[i]}")
    print(f"True class: {selected_labels[i]}")

In [ ]:
# Plot the predicted and true classes from tensorflow dataset for the two random picked images from the test set. 
plt.figure(figsize=(12, 9))

for i in range(len(selected_images)):
    ax = plt.subplot(len(selected_images), 1, i + 1)   #Create a subplot for each image
    plt.imshow(selected_images[i].astype("uint8")) #plot image
    color = 'green' if predicted_classes[i] == selected_labels[i] else 'red' #add color to true or false prediction
    plt.title(f"Predicted: {class_names[predicted_classes[i]]}, True: {class_names[selected_labels[i]]}", color=color)
    plt.axis("off")

**Adding more layers and modifications to the simple model** **Medium model**

In [ ]:
#medium model
medium_model = tf.keras.models.Sequential([
    tf.keras.layers.Rescaling(1./255),#rescale image to [0,1] range
    tf.keras.layers.RandomFlip(mode='horizontal_and_vertical'), #randomly flip images horizontally and vertically
    tf.keras.layers.Conv2D(32, 3, activation='relu'), #convolutional layer with 32 filters, kernel size 3 and relu activation function
    tf.keras.layers.Flatten(), #flatten the input shape
    tf.keras.layers.Dense(32, activation='relu'), #dense layer with 32 neurons and relu activation function
    tf.keras.layers.Dropout(0.20), #dropout layer with 20% dropout rate
    tf.keras.layers.Dense(8, activation='relu'), #dense layer with 64 neurons and relu activation function
    tf.keras.layers.Dense(len(class_names)),#dense layer with number of neurons equal 5 classes
    
])
# dummy input to create model summary
dummy_input = tf.random.normal(shape=(1, 128, 128, 3)) #create dummy input for model

medium_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
) #compile model with optimizer, loss function and metrics

# Call the model with a dummy input to trigger the build process
medium_model(dummy_input) 

# Now you can print the summary
medium_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Define early stopping but not relevant when epochs are set to 50 for all models. But can be used in order to stop training when validation loss does not improve anymore.
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=5,  # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value
)

#fit model to training data and validate on validation data
medium_modelhistory = medium_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=epochs,
    callbacks=[early_stopping]
) #fit model to training data and validate on validation data

#save the model
medium_model.save('medium_model.h5')

In [ ]:
#accuary and loss plot
medium_model_metrics = medium_modelhistory.history #extract metrics from history
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.plot(medium_modelhistory.epoch, medium_model_metrics['loss'], medium_model_metrics['val_loss'])
plt.legend(['training', 'validation'])
plt.ylim([0, max(plt.ylim())])
plt.ylabel('Loss')
plt.xlabel('Epoch')

plt.subplot(1, 2, 2)
plt.plot(medium_modelhistory.epoch, medium_model_metrics['accuracy'], medium_model_metrics['val_accuracy'])
plt.legend(['training', 'validation'])
plt.ylim([0, 1])
plt.ylabel('Accuracy')
plt.xlabel('Epoch')

In [ ]:
#evaluate model on test data and print confusion matrix
medium_model.evaluate(test_ds, return_dict=True) #evaluate model on test data

y_pred_medium = medium_model.predict(test_ds) #predict labels for test data
y_pred_medium = np.argmax(y_pred_medium, axis=1) #extract predicted labels from test data
y_true_medium = np.concatenate([y for x, y in test_ds], axis=0) #extract true labels from test data

confusion_mtx = tf.math.confusion_matrix(y_true_medium, y_pred_medium) #create confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_mtx,
            xticklabels=class_names,
            yticklabels=class_names,
            annot=True, fmt='g')
plt.xlabel('Prediction')
plt.ylabel('Label')
plt.show()



print(classification_report(y_true_medium, y_pred_medium, target_names=class_names))

In [ ]:
#plot of 9 random images from test set with predicted and true label
plt.figure(figsize=(12, 9))
for images, labels in test_ds.take(1): #take one image and label from test set
    for i in range(9): #iterate over 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8")) #plot image
        color = 'green' if y_pred_medium[i] == labels[i] else 'red' #if predicted label is the same as true label, color is green, else red
        plt.title(f"Predicted: {class_names[y_pred_medium[i]]}, True: {class_names[labels[i]]}", color=color) #add predicted and true label to title
        plt.suptitle("Sugarcane Leafs (Healthy or Diseased)")
        plt.axis("off")

plt.show()

In [ ]:
import shap
import numpy as np

#TensorFlow dataset to numpy array
test_ds_numpy = np.concatenate([x for x, _ in test_ds], axis=0)
test_labels = np.concatenate([y for _, y in test_ds], axis=0)  # Get the true labels

#two images from the test set
selected_images = test_ds_numpy[8:10]
selected_labels = test_labels[8:10]  # Select the true labels for the selected images

#predictions for the selected images
predictions = medium_model.predict(selected_images)
predicted_classes = np.argmax(predictions, axis=1)  # Get the predicted classes

#shap explainer for the model predictions for selected_images
explainer = shap.GradientExplainer(medium_model, np.concatenate([x.numpy() for x, _ in train_ds.take(2)], axis=0))

#shap values from explainer
shap_values, indexes = explainer.shap_values(selected_images, ranked_outputs=len(class_names))

#Plot shap values
shap.image_plot(shap_values, selected_images,show=False)


#Print the predicted and true classes
for i in range(len(selected_images)):
    print(f"Image {i+1}:")
    print(f"Predicted class: {predicted_classes[i]}")
    print(f"True class: {selected_labels[i]}")

In [ ]:
# Plot the predicted and true classes from tensorflow dataset for test set for the two images 
plt.figure(figsize=(12, 9))

for i in range(len(selected_images)):
    ax = plt.subplot(len(selected_images), 1, i + 1)
    plt.imshow(selected_images[i].astype("uint8"))
    color = 'green' if predicted_classes[i] == selected_labels[i] else 'red'
    plt.title(f"Predicted: {class_names[predicted_classes[i]]}, True: {class_names[selected_labels[i]]}", color=color)
    plt.axis("off")

**Large model including pretrained model MobileNetV2**

In [ ]:
#large model

mobile_model = tf.keras.applications.MobileNetV2(
    input_shape=input_shape,
    include_top=False,
    weights='imagenet'
) #import pretrained model
mobile_model.trainable = True #set model to trainable

print(len(mobile_model.layers)) #print number of layers in model
for layer in mobile_model.layers[:fine_tune_point]: #freeze layers up to fine tune point
 layer.trainable = False  #set layers to not trainable


#dummy data to print summary
dummy_input = tf.random.normal(shape=(1, 128, 128, 3)) #create dummy input for model


large_model = tf.keras.models.Sequential([
    tf.keras.layers.Rescaling(1./127.5, offset=-1), #rescale image values to 0-1
    #tf.keras.layers.RandomContrast(factor=0.1),
    tf.keras.layers.RandomFlip(mode='horizontal_and_vertical'), #randomly flip images horizontally and vertically
    mobile_model, #import mobilenetv2 model
    tf.keras.layers.GlobalAveragePooling2D(), #pooling layer to reduce dimensionality
    tf.keras.layers.Flatten(), #flatten layer to convert 2D to 1D array
    tf.keras.layers.Dense(64, activation='relu'), #dense layer with 64 neurons and relu activation function
    tf.keras.layers.Dropout(0.50), #dropout layer to prevent overfitting
    tf.keras.layers.Dense(len(class_names)) #dense layer with number of neurons equal to number of classes
]) #create model with sequential layers
# model.build(input_shape=(1, 128, 128, 3)) #build model with input shape


large_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
) #compile model with optimizer, loss function and metrics

#call the model for model summary
large_model(dummy_input) 

#print the summary
large_model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping


# Define early stopping but not relevant when epochs are set to 50 for all models. But can be used in order to stop training when validation loss does not improve anymore.
early_stopping = EarlyStopping(
    monitor='val_loss',  # Monitor validation loss
    patience=5,  # Number of epochs with no improvement after which training will be stopped
    restore_best_weights=True  # Restore model weights from the epoch with the best value
)

large_modelhistory = large_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=50,
    callbacks=[early_stopping]
) #fit model to training data and validate on validation data

#save the model
large_model.save('large_model.h5')

In [ ]:
#export the metrics from the history of large model
large_modelmetrics = large_modelhistory.history #extract metrics from history
plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.plot(large_modelhistory.epoch, large_modelmetrics['loss'], large_modelmetrics['val_loss'])
plt.legend(['training', 'validation'])
plt.ylim([0, max(plt.ylim())])
plt.ylabel('Loss')
plt.xlabel('Epoch')

plt.subplot(1, 2, 2)
plt.plot(large_modelhistory.epoch, large_modelmetrics['accuracy'], large_modelmetrics['val_accuracy'])
plt.legend(['training', 'validation'])
plt.ylim([0, 1])
plt.ylabel('Accuracy')
plt.xlabel('Epoch')

In [ ]:
 
large_model.evaluate(test_ds, return_dict=True) #evaluate model on test data

y_pred_large = large_model.predict(test_ds) #predict labels for test data
y_pred_large = np.argmax(y_pred_large, axis=1) #extract predicted labels from test data
y_true_large = np.concatenate([y for x, y in test_ds], axis=0) #extract true labels from test data

confusion_mtx = tf.math.confusion_matrix(y_true_large, y_pred_large) #create confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_mtx,
            xticklabels=class_names,
            yticklabels=class_names,
            annot=True, fmt='g')
plt.xlabel('Prediction')
plt.ylabel('Label')
plt.show()



print(classification_report(y_true_large, y_pred_large, target_names=class_names))

In [ ]:
#image visualisaiton of the model predictions. And the true labels of the images and add the true labels to the plot title

# plt.figure(figsize=(12, 9))
# for images, labels in test_ds.take(1): #take one image and label from test set
#     for i in range(9): #iterate over 9 images
#         ax = plt.subplot(3, 3, i + 1)
#         plt.imshow(images[i].numpy().astype("uint8")) #plot image
#         plt.title(f"Predicted: {class_names[y_pred[i]]}, True: {class_names[labels[i]]}")
#         plt.suptitle("Sugarcane Leafs (Healthy or Diseased)")
#         plt.axis("off")

# plt.show()

plt.figure(figsize=(12, 9))
for images, labels in test_ds.take(1): #take one image and label from test set
    for i in range(9): #iterate over 9 images
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8")) #plot image
        color = 'green' if y_pred_large[i] == labels[i] else 'red' #if predicted label is the same as true label, color is green, else red
        plt.title(f"Predicted: {class_names[y_pred_large[i]]}, True: {class_names[labels[i]]}", color=color) #add predicted and true label to title
        plt.suptitle("Sugarcane Leafs (Healthy or Diseased)")
        plt.axis("off")

plt.show()

In [ ]:
import shap
import numpy as np

#TensorFlow dataset to numpy array
test_ds_numpy = np.concatenate([x for x, _ in test_ds], axis=0)
test_labels = np.concatenate([y for _, y in test_ds], axis=0)  # Get the true labels

#two images from the test set
selected_images = test_ds_numpy[8:10]
selected_labels = test_labels[8:10]  # Select the true labels for the selected images

#predictions for the selected images
predictions = large_model.predict(selected_images)
predicted_classes = np.argmax(predictions, axis=1)  # Get the predicted classes

#shap explainer for the model predictions for selected_images
explainer = shap.GradientExplainer(large_model, np.concatenate([x.numpy() for x, _ in train_ds.take(2)], axis=0))

#shap values from explainer
shap_values, indexes = explainer.shap_values(selected_images, ranked_outputs=5)

#Plot shap values
shap.image_plot(shap_values, selected_images,show=False)


#Print the predicted and true classes
for i in range(len(selected_images)):
    print(f"Image {i+1}:")
    print(f"Predicted class: {predicted_classes[i]}")
    print(f"True class: {selected_labels[i]}")

In [ ]:
# Plot the predicted and true classes from tensorflow dataset for the two images
plt.figure(figsize=(12, 9))

for i in range(len(selected_images)):
    ax = plt.subplot(len(selected_images), 1, i + 1)
    plt.imshow(selected_images[i].astype("uint8"))
    color = 'green' if predicted_classes[i] == selected_labels[i] else 'red'
    plt.title(f"Predicted: {class_names[predicted_classes[i]]}, True: {class_names[selected_labels[i]]}", color=color)
    plt.axis("off")

# Part B — Bioethanol material balance and utility screening
## Scope and assumptions
Juice reception → fermentation → ethanol recovery. Upstream juice extraction, sucrose inversion, biomass growth, by-products, pretreatment, utilities generation and wastewater treatment are outside this simplified boundary.

Basis: 1,000 kg/h of glucose-equivalent aqueous feed. Glucose reaction: C6H12O6 → 2 C2H5OH + 2 CO2. Using rounded molecular weights, ethanol/glucose = 92/180 and CO2/glucose = 88/180. Unconverted glucose remains in the residue. Recovered product is expressed as **pure-ethanol equivalent**, not a claim that a column produces anhydrous fuel.

Every default below is an **illustrative assumption**, not fitted, measured, or taken from BPCL/HPCL. Recovery and specific recovery heat must eventually come from experiment or a rigorous separation simulation. The energy calculation includes feed sensible heating plus an assumed recovery duty; it is not a complete enthalpy balance. Heat recovery is an assumed fraction, not a pinch-analysis result.

| Parameter | Default | Meaning |
|---|---:|---|
| Feed | 1,000 kg/h | Aqueous glucose equivalent |
| Sugar mass fraction | 0.16 | kg sugar/kg feed |
| Conversion | 0.90 | Fraction of sugar reacted |
| Ethanol recovery | 0.98 | Fraction of formed ethanol recovered |
| Feed / target temperature | 25 / 32 °C | Sensible heat only |
| Feed heat capacity | 4.0 kJ/(kg K) | Constant assumption |
| Recovery heat | 8 MJ/kg recovered ethanol | Screening coefficient |
| Heat-recovery fraction | 0.25 | Applied to gross heat estimate |
| Useful steam heat | 2.1 MJ/kg | Assumed enthalpy drop |
| Steam cost | 2 INR/kg | Scenario; not a current quotation |
| Utility emission factor | 0.07 kg CO2e/MJ | Scenario; no lifecycle claim |

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, replace

@dataclass(frozen=True)
class Case:
    feed_kg_h: float = 1000.0
    sugar_fraction: float = 0.16
    conversion: float = 0.90
    recovery: float = 0.98
    feed_C: float = 25.0
    target_C: float = 32.0
    cp_kJ_kgK: float = 4.0
    recovery_MJ_kg: float = 8.0
    heat_recovery: float = 0.25
    steam_MJ_kg: float = 2.1
    steam_INR_kg: float = 2.0
    utility_kgCO2e_MJ: float = 0.07

def simulate(c):
    for key in ('sugar_fraction', 'conversion', 'recovery', 'heat_recovery'):
        if not 0 <= getattr(c, key) <= 1:
            raise ValueError(key + ' must lie between 0 and 1')
    if c.feed_kg_h <= 0 or c.steam_MJ_kg <= 0:
        raise ValueError('Feed and useful steam enthalpy must be positive')
    if min(c.cp_kJ_kgK, c.recovery_MJ_kg, c.steam_INR_kg, c.utility_kgCO2e_MJ) < 0:
        raise ValueError('Physical/cost coefficients cannot be negative')
    sugar = c.feed_kg_h*c.sugar_fraction
    reacted = sugar*c.conversion
    formed = reacted*92/180
    co2 = reacted*88/180
    product = formed*c.recovery
    residue = c.feed_kg_h-co2-product
    sensible = c.feed_kg_h*c.cp_kJ_kgK*max(c.target_C-c.feed_C, 0)/1000
    recovery_heat = product*c.recovery_MJ_kg
    net_heat = (sensible+recovery_heat)*(1-c.heat_recovery)
    steam = net_heat/c.steam_MJ_kg
    return dict(ethanol_kg_h=product, CO2_kg_h=co2, residue_kg_h=residue,
                unconverted_sugar_kg_h=sugar-reacted,
                ethanol_loss_kg_h=formed-product,
                mass_residual_kg_h=c.feed_kg_h-(product+co2+residue),
                sensible_MJ_h=sensible, recovery_heat_MJ_h=recovery_heat,
                net_heat_MJ_h=net_heat, steam_kg_h=steam,
                steam_cost_INR_h=steam*c.steam_INR_kg,
                utility_emissions_kgCO2e_h=net_heat*c.utility_kgCO2e_MJ,
                heat_MJ_kg_ethanol=net_heat/product if product else np.nan)
base = Case()
print(pd.Series(simulate(base)).round(4).to_string())

ethanol_kg_h                   72.1280
CO2_kg_h                       70.4000
residue_kg_h                  857.4720
unconverted_sugar_kg_h         16.0000
ethanol_loss_kg_h               1.4720
mass_residual_kg_h              0.0000
sensible_MJ_h                  28.0000
recovery_heat_MJ_h            577.0240
net_heat_MJ_h                 453.7680
steam_kg_h                    216.0800
steam_cost_INR_h              432.1600
utility_emissions_kgCO2e_h     31.7638
heat_MJ_kg_ethanol              6.2911


## Verification
These checks test reaction stoichiometry, component accounting, limiting cases and response to heat recovery. Algebraic closure alone does not validate the process physically. Fermentation CO2 is reported separately from utility emissions; it is not silently counted as fossil CO2 or treated as a credit.

In [2]:
r = simulate(base)
assert abs(r['mass_residual_kg_h']) < 1e-9
assert np.isclose(r['ethanol_kg_h']+r['ethanol_loss_kg_h'], 1000*.16*.9*92/180)
assert np.isclose(r['residue_kg_h'], 1000*(1-.16)+r['unconverted_sugar_kg_h']+r['ethanol_loss_kg_h'])
assert simulate(replace(base, conversion=0))['ethanol_kg_h'] == 0
assert simulate(replace(base, sugar_fraction=0))['CO2_kg_h'] == 0
assert simulate(replace(base, recovery=0))['ethanol_kg_h'] == 0
assert simulate(replace(base, heat_recovery=.5))['steam_kg_h'] < r['steam_kg_h']
assert np.isclose(simulate(replace(base, feed_kg_h=2000))['ethanol_kg_h'],2*r['ethanol_kg_h'])
try:
    simulate(replace(base, conversion=1.1))
    raise AssertionError('Invalid conversion accepted')
except ValueError:
    pass
print('9 engineering checks passed.')

9 engineering checks passed.


## Sensitivity and uncertainty
First vary one assumption at a time. Then propagate independent uniform scenario ranges; these percentiles are **scenario uncertainty**, not statistical confidence intervals. Independence and ranges are unvalidated assumptions. The two largest uncertainties to resolve are actual fermentation performance and separation heat consumption.

In [3]:
rows=[]
for name, values in {'sugar_fraction':[.12,.16,.20], 'conversion':[.80,.90,.95],
                     'recovery_MJ_kg':[6.,8.,10.], 'heat_recovery':[0.,.25,.40]}.items():
    for value in values:
        rows.append(dict(parameter=name, value=value, **simulate(replace(base, **{name:value}))))
sensitivity=pd.DataFrame(rows)
print(sensitivity[['parameter','value','ethanol_kg_h','steam_kg_h','heat_MJ_kg_ethanol']].round(3).to_string(index=False))
rng=np.random.default_rng(42)
trials=[]
for _ in range(2000):
    c=replace(base, sugar_fraction=rng.uniform(.12,.20), conversion=rng.uniform(.80,.95),
              recovery_MJ_kg=rng.uniform(6,10), heat_recovery=rng.uniform(.15,.35))
    trials.append(simulate(c))
uncertainty=pd.DataFrame(trials)
print('Scenario percentiles (5th, 50th, 95th):')
print(uncertainty[['ethanol_kg_h','steam_kg_h','steam_cost_INR_h']].quantile([.05,.5,.95]).round(2).to_string())

     parameter  value  ethanol_kg_h  steam_kg_h  heat_MJ_kg_ethanol
sugar_fraction   0.12        54.096     164.560               6.388
sugar_fraction   0.16        72.128     216.080               6.291
sugar_fraction   0.20        90.160     267.600               6.233
    conversion   0.80        64.114     193.182               6.328
    conversion   0.90        72.128     216.080               6.291
    conversion   0.95        76.135     227.529               6.276
recovery_MJ_kg   6.00        72.128     164.560               4.791
recovery_MJ_kg   8.00        72.128     216.080               6.291
recovery_MJ_kg  10.00        72.128     267.600               7.791
 heat_recovery   0.00        72.128     288.107               8.388
 heat_recovery   0.25        72.128     216.080               6.291
 heat_recovery   0.40        72.128     172.864               5.033
Scenario percentiles (5th, 50th, 95th):
      ethanol_kg_h  steam_kg_h  steam_cost_INR_h
0.05         53.29      143

## Constrained operating-point screening
A deliberately simplified hypothesis makes conversion depend on residence time: X = 1 − exp(−k τ), k=0.04 h⁻¹. This is an **unfitted first-order approximation**, not validated microbial kinetics. At density 1,000 kg/m³, V = feed mass flow × residence time / density. A 45 m³ vessel limit and minimum ethanol output of 65 kg/h constrain the search.

Search the grid for minimum utility cost per kg of recovered ethanol. Heat recovery is fixed so the optimiser cannot claim free energy savings. No capital cost, residence-time-dependent cooling, ethanol inhibition or contamination penalty is included; the optimum may lie on the capacity boundary. Report it as a constrained screening result, not a commercial optimum.

In [4]:
candidates=[]
for tau in np.linspace(12,60,193):
    conversion=1-np.exp(-.04*tau)
    v=base.feed_kg_h/1000*tau
    result=simulate(replace(base, conversion=conversion))
    if v <= 45 and result['ethanol_kg_h'] >= 65:
        candidates.append(dict(residence_h=tau, volume_m3=v, conversion=conversion,
                               utility_INR_kg=result['steam_cost_INR_h']/result['ethanol_kg_h'], **result))
if not candidates:
    print('No feasible point: change capacity or output requirements, not the result.')
else:
    feasible=pd.DataFrame(candidates)
    best=feasible.loc[feasible.utility_INR_kg.idxmin()]
    assert best.volume_m3 <=45 and best.ethanol_kg_h >=65
    print('Best feasible grid point under illustrative assumptions:')
    print(best[['residence_h','volume_m3','conversion','ethanol_kg_h','utility_INR_kg']].round(4).to_string())

Best feasible grid point under illustrative assumptions:
residence_h       45.0000
volume_m3         45.0000
conversion         0.8347
ethanol_kg_h      66.8948
utility_INR_kg     6.0133


## What makes this interview-ready
1. Obtain at least one documented fermentation dataset: sugar concentration, temperature, pH, time, ethanol titre and yield. Fit kinetics and compare predictions on held-out batches. Do not generate synthetic data and call it experimental validation.
2. Replace the recovery heat coefficient with a validated ethanol–water separation model. Discuss azeotropy and molecular-sieve dehydration; report final purity and recovery separately.
3. Add stream temperatures and heat capacities for a real pinch analysis, checking minimum approach temperature and feasible exchanger matches. Compare hot/cold utility targets against the base flowsheet.
4. Extend the boundary to fermentation cooling, water recycle and wastewater COD. Add a preliminary hazard discussion: ethanol flammability, CO2 accumulation, pressure/vacuum and contamination. This does not constitute a certified HAZOP.
5. For a 2G bagasse extension, explicitly add dry-basis cellulose/hemicellulose/lignin, moisture, pretreatment, hydrolysis, inhibitors and pentose fermentation. Do not describe the current juice model as 2G.
6. Only connect image screening to feedstock decisions after collecting paired leaf-image and juice-quality data. Use plant/farm/season-separated test sets; SHAP is explanatory evidence, not causality.

**Candidate title:** Bioethanol Process Modelling and Utility Sensitivity Analysis.

**Resume wording to adapt only after running and understanding your contribution:** Developed a Python screening model for glucose-equivalent bioethanol production, implementing stoichiometric balances, utility estimation, scenario uncertainty analysis and capacity-constrained operating-point selection. Distinguished assumed parameters from experimental validation and identified separation and heat-integration improvements.

Do not claim refinery deployment, plant savings, industrial accuracy, or authorship of the original CNN study. All new process results are illustrative.

## Sources and next reading
- [Keras MobileNet documentation](https://keras.io/2/api/applications/mobilenet/) — required MobileNetV2 input scaling.
- [NREL 2011 process design report, NREL/TP-5100-47764](https://afdc.energy.gov/files/pdfs/47764.pdf) — reference for a future lignocellulosic process extension; its corn-stover assumptions are not substituted for sugarcane data here.
- [BPCL environmental disclosures](https://www.bharatpetroleum.in/sustainability/environmental-clearance) — industrial ethanol context.
- [HPCL regulatory disclosures](https://www.hindustanpetroleum.com/pages/Regulatory-Requirements) — Bathinda 2G ethanol context.

## Execution status
Part B was executed and checked during this revision. Part A was reviewed and patched but not trained or evaluated: the images and model weights were not supplied. No improved classifier metric is claimed.